# Week 4 Guided Lab: GitHub, Reproducibility, Governance, and Ethics

**BAN 6003: Data Management and Analytics Integration**

This lab is intentionally light on new coding. The goal is to improve your professional workflow: repository organization, readable notebook structure, clean Python style, data dictionary drafting, and responsible handling of sensitive fields.

## Learning Goals

By the end of this lab, you should be able to:

- Use a simple reproducible project structure.
- Write clearer Python using descriptive names and reusable paths.
- Organize a notebook so its purpose, inputs, checks, decisions, and outputs are visible.
- Draft and save a useful data dictionary.
- Distinguish direct identifiers, confidential identifiers, quasi-identifiers, and sensitive business fields.
- Create and verify an audience-specific review copy.
- Explain responsible AI and tool use when data may be sensitive.

Weeks 3 and 4 use the same protected raw employee dataset. Week 3 focuses on cleaning decisions; Week 4 focuses on documentation, reproducibility, and responsible sharing.

## Part 1: Start with imports and paths

A reproducible notebook should make it clear where the data comes from and where outputs will go. We will use `pathlib.Path` instead of hard-coding long computer-specific file paths.

In [2]:
from pathlib import Path
import pandas as pd

# Project-style paths
RAW_DATA_PATH = Path("../data/raw/employee_data.csv")
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError(f"Required raw input was not found: {RAW_DATA_PATH}")

print("Raw data path:", RAW_DATA_PATH)
print("Processed data folder:", PROCESSED_DATA_DIR)

Raw data path: ../data/raw/employee_data.csv
Processed data folder: ../data/processed


### Why this matters

Avoid paths like `C:/Users/yourname/Downloads/file.csv`. Those paths only work on your own machine. Relative paths make your notebook easier to run in GitHub Codespaces and easier for someone else to review.

## Part 2: Load the data with a descriptive variable name

Instead of naming every DataFrame `df`, use a name that communicates the role of the data.

In [3]:
employees_raw = pd.read_csv(RAW_DATA_PATH)
employees_raw.head()

,employee_id,first_name,last_name,hire_date,department,salary,age,full_time
0,101,Ava,Nguyen,2021-03-15,Finance,"$95,000",29,Yes
1,102,Ben,Carter,2020/07/01,Sales,78000,35,Y
2,103,Carla,Diaz,not a date,HR,62000,thirty,No
3,104,David,Evans,2019-11-30,IT,"$88,500",41,Yes
4,105,Ella,Ford,NaN,Marketing,NaN,27,No


In [4]:
employees_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   employee_id  13 non-null     str  
 1   first_name   13 non-null     str  
 2   last_name    13 non-null     str  
 3   hire_date    11 non-null     str  
 4   department   12 non-null     str  
 5   salary       12 non-null     str  
 6   age          12 non-null     str  
 7   full_time    13 non-null     str  
dtypes: str(8)
memory usage: 964.0 bytes


## Part 3: Clean code style in practice

Good code does not need to be complicated. It should be readable. A few habits help:

- Use `snake_case` for variable names.
- Use descriptive names like `employees_raw` or `salary_missing_count`.
- Put important paths or settings near the top.
- Use comments to explain why, not to repeat the obvious.

In the next cell, we compute a few quick profile values using clear variable names.

In [5]:
row_count, column_count = employees_raw.shape
salary_missing_count = employees_raw["salary"].isna().sum()
duplicate_employee_id_count = employees_raw["employee_id"].duplicated().sum()

print(f"Rows: {row_count}")
print(f"Columns: {column_count}")
print(f"Missing salary values: {salary_missing_count}")
print(f"Duplicate employee_id values: {duplicate_employee_id_count}")

Rows: 13
Columns: 8
Missing salary values: 1
Duplicate employee_id values: 1


### Your turn: improve names

The code below works, but the names are not very clear. Rewrite it using more descriptive variable names.

In [6]:
# Original version
x = employees_raw["age"].isna().sum()
y = employees_raw["department"].value_counts()

print(x)
print(y)

1
department
Finance      3
Sales        2
HR           2
IT           2
Marketing    2
sales        1
Name: count, dtype: int64


In [7]:
age_missing_count = employees_raw["age"].isna().sum()
department_counts = employees_raw["department"].value_counts()

print(age_missing_count)
print(department_counts)




1
department
Finance      3
Sales        2
HR           2
IT           2
Marketing    2
sales        1
Name: count, dtype: int64


## Part 4: Notebook organization

A professional notebook should tell a story. A useful structure is:

1. Purpose
2. Inputs
3. Data checks
4. Decisions or observations
5. Outputs

Before you run a block of code, tell the reader what you are checking. After the code, write what you found.

### Practice: turn output into an observation

Run the next cell, then write 3-4 sentences in the markdown cell below. Report at least two results with numbers. Also explain one limitation of this check: values such as `"invalid"` or `"not a date"` are not counted as missing until they are converted or validated.

In [8]:
employees_raw.isna().sum()

employee_id    0
first_name     0
last_name      0
hire_date      2
department     1
salary         1
age            1
full_time      0
dtype: int64

**Your observation:**

The check shows that almost all columns are complete. There are 2 missing values in the hire_date column. There is also 1 missing value in the department column and 1 missing value in the salary column. There were no missing values found in the employee_id, first_name, last_name, or full_time columns. One limitation of using isna().sum() is that it only identifies actual missing values and does not identify invalid entries like incorrect dates, misspelling, and more.

## Part 5: Build a practical data dictionary

A data dictionary helps another person interpret and govern the data. For this lab, it includes:

- `column_name`
- `current_dtype`
- `business_description`
- `sensitivity_level`
- `cleaning_or_governance_notes`
- `sharing_guidance`

The final column turns classification into an action: who may use the field, or what protection is needed before sharing.

In [9]:
data_dictionary = pd.DataFrame({
    "column_name": employees_raw.columns,
    "current_dtype": [str(dtype) for dtype in employees_raw.dtypes],
    "business_description": [
        "Internal employee identifier",
        "Employee first name",
        "Employee last name",
        "Employee hire date",
        "Employee department",
        "Employee salary",
        "Employee age",
        "Whether the employee works full time"
    ],
    "sensitivity_level": [
        "Confidential internal identifier",
        "Direct identifier / PII",
        "Direct identifier / PII",
        "Quasi-identifier",
        "Business attribute",
        "Sensitive financial/business information",
        "Quasi-identifier",
        "Employment attribute"
    ],
    "cleaning_or_governance_notes": [
        "Should be unique; validate non-numeric and duplicate values",
        "Not needed for most analytical summaries",
        "Not needed for most analytical summaries",
        "Parse and validate dates before analysis",
        "Clarify valid categories and business definitions",
        "Convert to numeric and investigate missing or invalid values",
        "Convert to numeric and investigate impossible values",
        "Clarify meaning and standardize valid values"
    ],
    "sharing_guidance": [
        "Restrict to authorized users who need record-level traceability",
        "Remove unless the approved purpose requires a name",
        "Remove unless the approved purpose requires a name",
        "Generalize or remove for broad sharing",
        "Review disclosure risk when combined with other fields",
        "Restrict access; aggregate for broad audiences",
        "Group into ranges or remove for broad sharing",
        "Use only when relevant to the approved purpose"
    ]
})

data_dictionary

,column_name,current_dtype,business_description,sensitivity_level,cleaning_or_governance_notes,sharing_guidance
0,employee_id,str,Internal employee identifier,Confidential internal identifier,Should be unique; validate non-numeric and dup...,Restrict to authorized users who need record-l...
1,first_name,str,Employee first name,Direct identifier / PII,Not needed for most analytical summaries,Remove unless the approved purpose requires a ...
2,last_name,str,Employee last name,Direct identifier / PII,Not needed for most analytical summaries,Remove unless the approved purpose requires a ...
3,hire_date,str,Employee hire date,Quasi-identifier,Parse and validate dates before analysis,Generalize or remove for broad sharing
4,department,str,Employee department,Business attribute,Clarify valid categories and business definitions,Review disclosure risk when combined with othe...
5,salary,str,Employee salary,Sensitive financial/business information,Convert to numeric and investigate missing or ...,Restrict access; aggregate for broad audiences
6,age,str,Employee age,Quasi-identifier,Convert to numeric and investigate impossible ...,Group into ranges or remove for broad sharing
7,full_time,str,Whether the employee works full time,Employment attribute,Clarify meaning and standardize valid values,Use only when relevant to the approved purpose


### Your turn: improve the data dictionary

Edit at least two entries in `business_description`, `cleaning_or_governance_notes`, or `sharing_guidance`. Make each edit specific enough that another analyst would know what the field means or how it should be handled.

In [12]:
data_dictionary.loc[
    data_dictionary["column_name"] == "salary",
    "sharing_guidance"
] = "Only HR and managers should have access to salary information."

data_dictionary.loc[
    data_dictionary["column_name"] == "hire_date",
    "cleaning_or_governance_notes"
] = "Check for missing dates and make sure all dates use the same format."


In [13]:
dictionary_output_path = PROCESSED_DATA_DIR / "employee_data_dictionary.csv"
data_dictionary.to_csv(dictionary_output_path, index=False)
print("Saved data dictionary to:", dictionary_output_path)

Saved data dictionary to: ../data/processed/employee_data_dictionary.csv


Saving the dictionary makes the documentation part of the reproducible output, rather than leaving it only as a temporary object in memory.

## Part 6: Identify PII and sensitive fields

PII means personally identifiable information. Some columns identify a person directly. Other columns may become identifying when combined with other fields. Some fields may not identify a person, but still carry business or ethical risk.

In [14]:
sensitivity_summary = data_dictionary[["column_name", "sensitivity_level"]]
sensitivity_summary

,column_name,sensitivity_level
0,employee_id,Confidential internal identifier
1,first_name,Direct identifier / PII
2,last_name,Direct identifier / PII
3,hire_date,Quasi-identifier
4,department,Business attribute
5,salary,Sensitive financial/business information
6,age,Quasi-identifier
7,full_time,Employment attribute


### Reflection: privacy and governance

Answer the following questions in 4-6 sentences:

1. Which columns are direct identifiers?
2. Which columns remain sensitive or potentially identifying even after names are removed?
3. For a broad management audience, which fields would you remove, generalize, or aggregate first?
4. Why is the intended audience and business purpose part of the release decision?

**Your response:**

1. The direct identifiers in this dataset are first_name and last_name because they can identify a specific employee. 2. employee_id, hire_date, age, and salary could still be sensitive or help identify someone when compared with other information. 3. I would remove names and employee IDs and group age and salary into ranges instead of exact values. 4. The intended audience and business purpose matter because not everyone needs access to sensitive employee information, and limiting data helps protect privacy.

## Part 7: Create a limited internal review copy

Scenario: an authorized HR data-quality reviewer needs row-level records and employee IDs to trace possible source-system issues, but does not need employee names. Create a limited review copy without `first_name` and `last_name`.

This file is **not anonymous**. It still contains a confidential employee ID, exact dates, age, and salary. It should remain restricted to the approved internal audience and should not be posted publicly or emailed broadly.

In [15]:
direct_name_columns = ["first_name", "last_name"]
employees_review_copy = employees_raw.drop(columns=direct_name_columns).copy()

# Verify that names were removed without changing the number of records.
assert len(employees_review_copy) == len(employees_raw)
assert set(direct_name_columns).isdisjoint(employees_review_copy.columns)
assert set(direct_name_columns).issubset(employees_raw.columns)

employees_review_copy.head()

,employee_id,hire_date,department,salary,age,full_time
0,101,2021-03-15,Finance,"$95,000",29,Yes
1,102,2020/07/01,Sales,78000,35,Y
2,103,not a date,HR,62000,thirty,No
3,104,2019-11-30,IT,"$88,500",41,Yes
4,105,NaN,Marketing,NaN,27,No


In [16]:
review_output_path = PROCESSED_DATA_DIR / "employee_review_copy_no_names.csv"
employees_review_copy.to_csv(review_output_path, index=False)
print("Saved restricted internal review copy to:", review_output_path)

Saved restricted internal review copy to: ../data/processed/employee_review_copy_no_names.csv


### Verify the saved review copy

Reload the file from disk and verify the actual saved artifact, not only the in-memory DataFrame. The checks below confirm that the row count is unchanged and direct names are absent.

In [17]:
saved_review_copy = pd.read_csv(review_output_path)

guided_release_checks = pd.Series({
    "source_rows": len(employees_raw),
    "saved_review_rows": len(saved_review_copy),
    "rows_match": len(saved_review_copy) == len(employees_raw),
    "first_name_present": "first_name" in saved_review_copy.columns,
    "last_name_present": "last_name" in saved_review_copy.columns,
    "employee_id_present": "employee_id" in saved_review_copy.columns,
    "salary_present": "salary" in saved_review_copy.columns,
})

guided_release_checks

source_rows               13
saved_review_rows         13
rows_match              True
first_name_present     False
last_name_present      False
employee_id_present     True
salary_present          True
dtype: object

## Part 8: Responsible AI and tool-use checkpoint

An AI assistant suggests uploading the employee CSV and asks to inspect it before recommending governance controls. In 3-4 sentences, explain why a real HR file should not be uploaded to an unapproved external tool, what information you would provide instead, and who remains accountable for the final decision.

HR files should not be uploaded to an unapproved external AI tool because it can contain personal/sensitive employee information. To avoid sensitive information to be exposed, I would provide a summary of the data, the column names, or a de-identified sample so no private information is shared. Regardless of the recommendation that the tool may provide, the actual person using the tool is responsible for reviewing the results and making the final decision.

## Part 9: GitHub workflow checkpoint

Save the notebook before using the terminal. Review `git status` so you understand which files will be committed, then use a meaningful message that describes the work.

```bash
git status
git add .
git commit -m "Complete Week 4 documentation and governance lab"
git push
```

## Part 10: Offline Assignment - Release-Readiness Audit

Complete this section independently after the guided lab.

Scenario: a teammate asks whether `employee_review_copy_no_names.csv` can be emailed to a broad group of managers because employee names were removed. Your job is to audit the saved file and make a defensible release decision.

1. Reload `review_output_path` as `offline_review`.
2. Complete the one-row `release_readiness_summary` started below. It checks source rows, review rows, whether row counts match, whether direct-name columns remain, and whether employee ID, salary, exact age, and exact hire date remain.
3. Display the summary.
4. Use the evidence to decide whether broad email distribution should be approved.

In [18]:
offline_review = pd.read_csv(review_output_path)

release_readiness_summary = pd.DataFrame([{
    "source_rows": len(employees_raw),
    "review_rows": len(offline_review),
    "rows_match": len(employees_raw) == len(offline_review),
    "direct_name_columns_present": bool(set(direct_name_columns) & set(offline_review.columns)),
    "employee_id_present": "employee_id" in offline_review.columns,
    "salary_present": "salary" in offline_review.columns,
    "exact_age_present": "age" in offline_review.columns,
    "exact_hire_date_present": "hire_date" in offline_review.columns
}])

release_readiness_summary



,source_rows,review_rows,rows_match,direct_name_columns_present,employee_id_present,salary_present,exact_age_present,exact_hire_date_present
0,13,13,True,False,True,True,True,True


### Release Decision

**Your response here:** 
I would not approve this file for broad email distribution. The row counts match and the direct name columns have been removed. Even with this removed, employee ID, salary, exact age, and exact hire date are still present in the file. These could be used to identify employees or reveal sensitive information. A safer option would be to share the file only with a limited group that needs access or provide a different version of the data with salary and age grouped into ranges.


## Final Submission Checklist

Before submitting, make sure:

- You ran the notebook from top to bottom without errors.
- You rewrote unclear variable names in Part 3.
- You wrote an evidence-based observation about missing values.
- You improved and saved the data dictionary.
- You completed the privacy/governance and responsible-tool responses.
- You created and verified the restricted internal review copy.
- You completed the offline release-readiness summary and decision.
- Both required files exist in `data/processed`.
- You reviewed `git status`, then committed and pushed your work.

## Optional Practice: Build and Audit a Purpose-Limited Copy (Not Graded)

This short practice does **not** count toward your lab grade. It gives you another chance to apply data minimization, clear paths, saved-artifact verification, and audience-based governance decisions.

Scenario: an approved internal manager needs to review only the recorded `department` and `full_time` categories. The manager does not need names, employee IDs, dates, salary, or age.


### Practice Task

1. Create `manager_category_copy` containing only `department` and `full_time` from `employees_raw`.
2. Save it to `PROCESSED_DATA_DIR / "optional_manager_category_copy.csv"`.
3. Reload the saved file as `saved_manager_category_copy`.
4. Create `optional_release_checks` that reports whether the row count matches the source, whether the saved file contains exactly the two approved columns, and whether any of these fields remain: `employee_id`, `first_name`, `last_name`, `hire_date`, `salary`, or `age`.

Hints:

- Select the two approved columns with `employees_raw[[...]].copy()`.
- Use `.to_csv(..., index=False)` and `pd.read_csv()` just as in the guided lab.
- Compare sets of column names to test the approved and restricted fields.


In [ ]:
# Optional practice workspace
# Build manager_category_copy, save it, reload it, and create optional_release_checks.
#
# approved_columns = ["department", "full_time"]
# restricted_columns = {"employee_id", "first_name", "last_name", "hire_date", "salary", "age"}


In [ ]:
# Run this self-check after you create saved_manager_category_copy and optional_release_checks.
required_practice_objects = {
    "saved_manager_category_copy",
    "optional_release_checks",
}

if not required_practice_objects.issubset(globals()):
    print("Complete the optional practice first, then rerun this self-check.")
else:
    expected_columns = {"department", "full_time"}
    restricted_columns = {"employee_id", "first_name", "last_name", "hire_date", "salary", "age"}
    print("Rows match source:", len(saved_manager_category_copy) == len(employees_raw))
    print("Only approved columns remain:", set(saved_manager_category_copy.columns) == expected_columns)
    print("Restricted columns present:", sorted(restricted_columns & set(saved_manager_category_copy.columns)))
    display(optional_release_checks)


### Think About the Result

Even if the file passes the column checks, release approval still depends on purpose, audience, data quality, and organizational policy. Notice that removing sensitive columns does not fix inconsistent category values or prove that broad distribution is appropriate. Be ready to explain what the technical checks establish and what still requires human judgment.
